In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("dengeli_veri_seti.csv")

In [3]:
df.head()

,id,score,review
0,47896,Neutral,Biraz kılları sökülüyor ama fena değil denemey...
1,48009,Positive,İlk kullanımla birlikte saçları yumuşacık yapi...
2,57013,Neutral,Sert biraz ama kullanıma engel değil
3,21824,Positive,her kes saçımın ne kadar güzel renke değiştiği...
4,125799,Neutral,Ürün gayet güzel


In [4]:
import string
import re
#re yani RegEx kütüphanesi kelimeleri tek tek aramak yerine rakam, sembol gibi desenleri tek satırda bulup silmemizi sağlar yani aynı fonksiyonda hem string hem de kolaylaştırılmış olan regex kullancaz

In [5]:
def metin_temizleme_hibrit(text):
    text = str(text)

    #python kafa karmaşası için ı ve i dönüşümü
    text = text.replace('İ', 'i').replace('I', 'ı')

    #küçük harfe çevirme
    text = text.lower()

    #noktalama işaretlerini silme
    #[^\w\s] harf ve boşluk haricindeki her şeyi emojiler de dahil siler
    text = re.sub(r'[^\w\s]', ' ', text)

    #rakamları silme (regex kütüphanesi ile) \d+ ile rakamları bul ve sil. burda \d herhangi bir rakam bbuluyor + olunca eğer o rakamın yanında başka rakamlar da varsa (örneğin 2026) hepsini bir bütün olarak alır. re.sub 3 parametresi var 1.aranacak desen, 2.yerine konacak metin, 3.kaynak metin
    text = re.sub(r'\d+', '', text)

    #boşlukları temizleme
    text = " ".join(text.split())

    return text


In [6]:
#tüm metinlere bu fonksiyonu uygulama
df['temiz_review'] = df['review'].apply(metin_temizleme_hibrit)

print("Temizleme bitti. sonuç:")
print(df[['review','temiz_review']].head(10))

Temizleme bitti. sonuç:
                                              review  \
0  Biraz kılları sökülüyor ama fena değil denemey...   
1  İlk kullanımla birlikte saçları yumuşacık yapi...   
2               Sert biraz ama kullanıma engel değil   
3  her kes saçımın ne kadar güzel renke değiştiği...   
4                                   Ürün gayet güzel   
5  orjinal ürün değil sanırım diğer achrominlerde...   
6         kollarının boyu çok uzun geldi bedeni uydu   
7  Erkek arkadaşımın ayak tırnaklarında sorun var...   
8  Ürün saçtan arınmıyor diyen arkadaşlar için;sa...   
9  altındaki lastik bantlar sayesinde hiç kayma y...   

                                        temiz_review  
0  biraz kılları sökülüyor ama fena değil denemey...  
1  ilk kullanımla birlikte saçları yumuşacık yapi...  
2               sert biraz ama kullanıma engel değil  
3  her kes saçımın ne kadar güzel renke değiştiği...  
4                                   ürün gayet güzel  
5  orjinal ürün değil sanırım

Stopwords keimeleri için nltk kütüphanesini indiriyoruz. bu kelimeler cümlede pek bir anlamı olmayan çıkarılması gereken bağlaç edat gibi kelimelerdir.

In [7]:
import nltk

In [8]:
from nltk.corpus import stopwords

In [9]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/omerayhan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [10]:
#Türkçe durdurma kelimelerini bir set içine alıyoruz
tr_stopwords = set(stopwords.words('turkish'))

#kendi özel durdurma kelimelerimizi de ekliyoruz
tr_stopwords.update(['bir','kadar','daha', 'çok', 'en'])

In [12]:
def stopwords_temizle(text):
    #önce cümleyi kelimelere ayırıyoruz
    kelimeler = text.split()

    #kelime eğer stopworld listesinde değilse onu temiz kelimeler listesine al
    temiz_kelimeler = [kelime for kelime in kelimeler if kelime not in tr_stopwords]

    return " ".join(temiz_kelimeler)


In [13]:
ornek_text = "ben bu telefonu çok beğendim ama kargosu çok yavaş geldi"
print("önceki hali = ",ornek_text)
print("sonraki hali: ",stopwords_temizle(ornek_text))

önceki hali =  ben bu telefonu çok beğendim ama kargosu çok yavaş geldi
sonraki hali:  ben telefonu beğendim kargosu yavaş geldi


In [14]:
#Türkçe yazım yanlışlarını java zemberek ile şuan düzeltemiyoruz. o yüzden kendi sözlüğümüzü oluşturduk şimdilik, bazı herleri silinmesin diye bilerek birleşik yazıyoruz, dict yapısında anahtar değer gibi tutuyoruz.
yazim_sozlugu = {
    "her kes": "herkes",
    "hiç bir": "hiçbir",
    "her şey": "herşey", 
    "beyendim": "beğendim",
    "malesef": "maalesef",
    "yanlız": "yalnız",
    "herkez": "herkes"
}
def yazim_duzelt(text):
    text = str(text)
    for hatali, dogru in yazim_sozlugu.items():
        #yukarda items() yazmasaydık sadece hatalı kısmı verecekti
        text = text.replace(hatali, dogru)
        #burada da hatali yerine dogru replace edildi.
    return text

print("manuel olarak girilen sözlükteki yazım yanlışları düzeltiliyor.")
df['temiz_review'] = df['temiz_review'].apply(yazim_duzelt)



#şimdi bu stopwords temizliğini veri setimizde uygulayalım
df['temiz_review'] = df['temiz_review'].apply(stopwords_temizle)

print("temizleme bitti, sonuçlar: ")
print(df[['review','temiz_review']].head(10))
print(df[['review','temiz_review']].tail(10))


manuel olarak girilen sözlükteki yazım yanlışları düzeltiliyor.
temizleme bitti, sonuçlar: 
                                              review  \
0  Biraz kılları sökülüyor ama fena değil denemey...   
1  İlk kullanımla birlikte saçları yumuşacık yapi...   
2               Sert biraz ama kullanıma engel değil   
3  her kes saçımın ne kadar güzel renke değiştiği...   
4                                   Ürün gayet güzel   
5  orjinal ürün değil sanırım diğer achrominlerde...   
6         kollarının boyu çok uzun geldi bedeni uydu   
7  Erkek arkadaşımın ayak tırnaklarında sorun var...   
8  Ürün saçtan arınmıyor diyen arkadaşlar için;sa...   
9  altındaki lastik bantlar sayesinde hiç kayma y...   

                                        temiz_review  
0  biraz kılları sökülüyor fena değil denemeye de...  
1  ilk kullanımla birlikte saçları yumuşacık yapi...  
2                   sert biraz kullanıma engel değil  
3     herkes saçımın güzel renke değiştiğini soruyor  
4               

SIRADA STEMMING YANİ GÖVDE BULMA VAR.

In [15]:
#Türkçe dil yapısı sondan eklemeli olduğu için İngilizce ve Avrupa dillerinin standart snowball yapısına uymuyor.
#Biz de bu eksikliği gidermek için yazılmış olan TurkishStammer kütüphanesini indiriyoruz


In [16]:
from TurkishStemmer import TurkishStemmer
stemmer = TurkishStemmer()

In [17]:
#kök bulma fonksiyonunu yazalım
def govde_bulma(text):
    kelimeler = str(text).split()

    #govdeleri alma
    govdeler = [stemmer.stem(kelime) for kelime in kelimeler]

    return " ".join(govdeler)

In [18]:
#test örnek
ornek = "gözlükçüden aldığım kitapları okuyunca çok duygulandım"

print("önceki hali: ", ornek)
print("sonraki hali: ", govde_bulma(ornek))

önceki hali:  gözlükçüden aldığım kitapları okuyunca çok duygulandım
sonraki hali:  gözlükçü aldık kitap okuy çok duygulan


In [19]:
#kendi veri setimizde gövde bulma işlemini uygulayalım.
df['temiz_review'] = df['temiz_review'].apply(govde_bulma)

print("veri setinde gövde bulma sonuçları: ")
print(df[['review','temiz_review']].head())

veri setinde gövde bulma sonuçları: 
                                              review  \
0  Biraz kılları sökülüyor ama fena değil denemey...   
1  İlk kullanımla birlikte saçları yumuşacık yapi...   
2               Sert biraz ama kullanıma engel değil   
3  her kes saçımın ne kadar güzel renke değiştiği...   
4                                   Ürün gayet güzel   

                                        temiz_review  
0  biraz kıl sökülüyor fena değil deneme değer re...  
1  ilk kullan birlik saç yumuşacık yapiyo harika ...  
2                      sert biraz kullan engel değil  
3             herkes saç güzel renk değiştik soruyor  
4                                   ürün gayet güzel  


In [20]:
#veriyi tekrar kaydedelim
df.to_csv("temizlenmis_veri_seti.csv", index=False)
print("veri seti başarıyla temizlendi ve kaydedildi")

veri seti başarıyla temizlendi ve kaydedildi
